# Benchmark Scores Are Pipeline-Dependent — Experiments

This notebook walks through the core empirical results of the paper.
All data was produced by the original-pipeline evaluation harness; no model re-runs are needed.

**Models evaluated:** GPT-5.4, Claude Sonnet 4.6, Gemma-4-31B, Qwen3.6-35B, Llama-3.3-70B,
GPT-OSS-20B, Primus-Nemotron-70B, Primus-Merged-8B, Foundation-Sec-8B, RedSage-Qwen3-8B-DPO  
**Benchmarks:** CTI-Bench, AthenaBench, SECURE, SecEval, CyberMetric, SecBench, MMLU-CS, RedSage-Bench

---

## Table of Contents

1. [Setup](#1-setup)
2. [Prompt Sensitivity — Zero-Shot vs Few-Shot vs CoT](#2-sensitivity)  
   2.1 [Average scores by mode](#2-1)  
   2.2 [Per-task sensitivity heatmap](#2-2)  
   2.3 [F4(E): Reasoning–extraction conflict case studies](#2-3)
3. [Cross-Benchmark Analysis](#3-cross)  
   3.1 [Score matrix](#3-1)  
   3.2 [PCA — score-level redundancy](#3-2)  
   3.3 [Kendall-τ — rank disagreement](#3-3)  
   3.4 [Ranking shifts under pipeline standardization](#3-4)
4. [F1(A): Logprob vs Generative Scoring on RedSage-Bench](#4-logprob)
5. [Failure Mode Examples](#5-failures)  
   5.1 [F2(I): Token-budget filter — SecEval](#5-1)  
   5.2 [F2(E): Denominator inflation — CTI-Bench RCM](#5-2)  
   5.3 [F3(E): Metric-direction mismatch — VSP](#5-3)

## 1. Setup <a id='1-setup'></a>

In [ ]:
import json, glob, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy.stats import kendalltau
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

OUTPUTS = Path('../outputs')

MODEL_LABELS = {
    'claude_sonnet_4_6':      'Claude Sonnet 4.6',
    'gpt_5_4':                'GPT-5.4',
    'gemma4_31b':             'Gemma-4-31B',
    'qwen3_35b':              'Qwen3.6-35B',
    'llama33_70b':            'Llama-3.3-70B',
    'gpt_oss_20b':            'GPT-OSS-20B',
    'nemotron_70b':           'Primus-Nemotron-70B',
    'llama_primus_merged':    'Primus-Merged-8B',
    'foundation_sec_8b':      'Foundation-Sec-8B',
    'redsage_qwen3_8b':       'RedSage-Qwen3-8B',
    'fanar2_27b':             'Fanar-2-27B',
}

# Benchmark groupings used throughout
BENCHMARK_OF = {
    'mcq': 'CTI-Bench', 'rcm': 'CTI-Bench', 'rcm_2021': 'CTI-Bench',
    'vsp': 'CTI-Bench', 'ate': 'CTI-Bench', 'cti_taa': 'CTI-Bench',
    'ckt': 'AthenaBench', 'athena_ate': 'AthenaBench', 'athena_rcm': 'AthenaBench',
    'athena_vsp': 'AthenaBench', 'rms': 'AthenaBench', 'taa': 'AthenaBench',
    'secure_maet': 'SECURE', 'secure_cwet': 'SECURE', 'secure_kcv': 'SECURE',
    'seceval': 'SecEval', 'cybermetric': 'CyberMetric', 'secbench': 'SecBench',
    'mmlu-cs': 'MMLU-CS',
    'redsage_frameworks': 'RedSage', 'redsage_generals': 'RedSage',
    'redsage_skills': 'RedSage', 'redsage_cli': 'RedSage', 'redsage_kali': 'RedSage',
}
print('Setup complete.')

---
## 2. Prompt Sensitivity — Zero-Shot vs Few-Shot vs CoT <a id='2-sensitivity'></a>

Each of the 10 models was evaluated under three prompt configurations while holding
all other pipeline components fixed (same model, same judge, same token budgets,
same decoding parameters). The question is how much the *prompt mode alone* moves scores.

In [ ]:
with open(OUTPUTS / 'sensitivity_eval' / 'scores.json') as f:
    raw = json.load(f)

records = []
for model, tasks in raw.items():
    if not isinstance(tasks, dict): continue
    for task, modes in tasks.items():
        if not isinstance(modes, dict): continue
        records.append({
            'model': model,
            'model_label': MODEL_LABELS.get(model, model),
            'task': task,
            'benchmark': BENCHMARK_OF.get(task, 'Other'),
            'zero_shot': modes.get('zero_shot'),
            'few_shot':  modes.get('few_shot'),
            'cot':       modes.get('cot'),
        })

df = pd.DataFrame(records).dropna()
df['cot_delta'] = df['cot'] - df['zero_shot']
df['fs_delta']  = df['few_shot'] - df['zero_shot']
print(f'{df["model"].nunique()} models, {df["task"].nunique()} tasks, {len(df)} rows')

### 2.1 Average scores by mode <a id='2-1'></a>

In [ ]:
avg = (
    df.groupby('model_label')[['zero_shot', 'few_shot', 'cot']]
    .mean().round(1)
    .sort_values('zero_shot', ascending=False)
)

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(avg)); w = 0.25
ax.bar(x - w, avg['zero_shot'], w, label='Zero-shot', color='#4C72B0')
ax.bar(x,     avg['few_shot'],  w, label='Few-shot',  color='#DD8452')
ax.bar(x + w, avg['cot'],       w, label='CoT',       color='#55A868')
ax.set_xticks(x)
ax.set_xticklabels(avg.index, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Mean accuracy (%) across all tasks')
ax.set_title('Average accuracy by prompt mode — macro view')
ax.legend(); plt.tight_layout(); plt.show()
print(avg.to_string())

### 2.2 Per-task sensitivity heatmap <a id='2-2'></a>

The macro averages above mask large task-level swings.
Each cell below shows the **CoT − zero-shot delta** (in percentage points).
Red = CoT hurts, blue = CoT helps.

In [ ]:
pivot = df.pivot_table(index='model_label', columns='task', values='cot_delta')
pivot = pivot.reindex(sorted(pivot.columns), axis=1)

fig, ax = plt.subplots(figsize=(20, 6))
sns.heatmap(
    pivot, cmap='RdBu_r', center=0, vmin=-70, vmax=70,
    annot=True, fmt='.0f', annot_kws={'size': 7},
    linewidths=0.3, ax=ax
)
ax.set_title('CoT vs zero-shot delta (pp). Red = CoT hurts, blue = CoT helps.')
ax.set_xlabel(''); ax.set_ylabel('')
plt.xticks(rotation=40, ha='right', fontsize=8)
plt.tight_layout(); plt.show()

### 2.3 F4(E): Reasoning–extraction conflict case studies <a id='2-3'></a>

**F4(E)** is a positional failure: the standard extractor reads only the **final line**.
Under CoT, models reason at length and often place correct answers inside the reasoning
trace rather than at the very end — so the extractor discards them.

The clearest cases in our data:
- `ckt`: Claude Sonnet 4.6 drops from 92 → 21 under CoT  
- `secbench`: Claude Sonnet 4.6 drops 88 → 34 under CoT  
- `ate` / `athena_ate`: CoT *helps* here because our unified harness uses an **LLM judge**
  for extraction (not a regex), so answers in the reasoning body are correctly identified.
  This asymmetry is the direct empirical signature of F4(E).

In [ ]:
focus = ['ckt', 'ate', 'athena_ate', 'secbench', 'rcm', 'rcm_2021']
sub = df[df['task'].isin(focus)]
modes = ['zero_shot', 'few_shot', 'cot']
mode_labels = ['ZS', 'FS', 'CoT']
colors = plt.cm.tab10.colors

fig, axes = plt.subplots(1, len(focus), figsize=(18, 5), sharey=False)
for ax, task in zip(axes, focus):
    tdf = sub[sub['task'] == task].reset_index(drop=True)
    for i, row in tdf.iterrows():
        ax.plot(mode_labels, [row[m] for m in modes],
                marker='o', color=colors[i % 10],
                label=row['model_label'], lw=1.5, ms=4)
    ax.set_title(task, fontsize=10)
    ax.set_ylim(0, 105)
    ax.set_ylabel('Accuracy (%)' if task == focus[0] else '')
    ax.grid(axis='y', alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=6, fontsize=8,
           bbox_to_anchor=(0.5, -0.14))
fig.suptitle('Prompt-mode trajectories on F4(E)-sensitive tasks', fontsize=12)
plt.tight_layout(); plt.show()

# Tabulate worst CoT drops
print('\nModels with CoT drop > 5pp per task:')
for task in focus:
    drops = df[(df['task'] == task) & (df['cot_delta'] < -5)]
    if len(drops):
        print(f'  {task}: {len(drops)} model(s), max drop {drops["cot_delta"].min():.1f}pp')
        for _, r in drops.iterrows():
            print(f'    {r["model_label"]}: {r["zero_shot"]:.0f} → {r["cot"]:.0f} ({r["cot_delta"]:+.0f}pp)')

---
## 3. Cross-Benchmark Analysis <a id='3-cross'></a>

We now ask whether the audited benchmarks support **stable model comparisons**.
We use zero-shot scores (the standardized mode) as a consistent baseline across all models and tasks.

In [ ]:
# Build a model × task score matrix from zero-shot scores
score_matrix = df.pivot_table(index='model_label', columns='task', values='zero_shot')
score_matrix = score_matrix.dropna(axis=1)  # keep tasks with all models
print(f'Score matrix: {score_matrix.shape[0]} models × {score_matrix.shape[1]} tasks')
score_matrix.round(1)

### 3.1 Score matrix heatmap <a id='3-1'></a>

In [ ]:
fig, ax = plt.subplots(figsize=(20, 5))
sns.heatmap(
    score_matrix.reindex(sorted(score_matrix.columns), axis=1),
    cmap='YlOrRd', annot=True, fmt='.0f', annot_kws={'size': 7},
    linewidths=0.3, vmin=0, vmax=100, ax=ax
)
ax.set_title('Zero-shot accuracy (%) per model × task')
ax.set_xlabel(''); ax.set_ylabel('')
plt.xticks(rotation=40, ha='right', fontsize=8)
plt.tight_layout(); plt.show()

### 3.2 PCA — score-level redundancy <a id='3-2'></a>

If all tasks measured independent capabilities, we would expect many PCA components
to contribute roughly equally to the variance. Instead, we find that a single component
dominates — consistent with the paper's finding that **PC1 explains 95.3% of variance**,
meaning most tasks primarily separate generally stronger from weaker models.

In [ ]:
X = StandardScaler().fit_transform(score_matrix.T)  # tasks as observations
pca = PCA()
pca.fit(X)
var_exp = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, len(var_exp) + 1), var_exp, color='#4C72B0')
ax.set_xlabel('Principal component')
ax.set_ylabel('Variance explained (%)')
ax.set_title(f'PCA scree plot — PC1 explains {var_exp[0]:.1f}% of variance')
ax.annotate(f'{var_exp[0]:.1f}%', xy=(1, var_exp[0]), xytext=(1.4, var_exp[0] - 5),
            fontsize=11, color='darkred', fontweight='bold')
plt.tight_layout(); plt.show()

print('Variance explained by component:')
for i, v in enumerate(var_exp[:5], 1):
    print(f'  PC{i}: {v:.1f}%  (cumulative: {var_exp[:i].sum():.1f}%)')

### 3.3 Kendall-τ — rank disagreement <a id='3-3'></a>

Even when tasks broadly agree on which models are stronger, they may disagree on the
relative ordering of specific models. We compute pairwise Kendall-τ between all task-induced
rankings. The paper highlights **VSP tasks (τ = 0.29)** and **TAA tasks (τ = 0.24)**
as cases where semantically similar tasks from CTI-Bench and AthenaBench disagree substantially.

In [ ]:
tasks = score_matrix.columns.tolist()
n = len(tasks)
tau_matrix = np.zeros((n, n))

for i, t1 in enumerate(tasks):
    for j, t2 in enumerate(tasks):
        tau, _ = kendalltau(score_matrix[t1].values, score_matrix[t2].values)
        tau_matrix[i, j] = tau

tau_df = pd.DataFrame(tau_matrix, index=tasks, columns=tasks)

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(tau_matrix, dtype=bool))
sns.heatmap(
    tau_df, mask=mask, cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    linewidths=0.3, ax=ax
)
ax.set_title('Pairwise Kendall-τ between task-induced model rankings')
plt.tight_layout(); plt.show()

print('Low-agreement task pairs (τ < 0.4):')
for i, t1 in enumerate(tasks):
    for j, t2 in enumerate(tasks):
        if i < j and tau_matrix[i, j] < 0.4:
            print(f'  {t1} vs {t2}: τ = {tau_matrix[i, j]:.2f}')

### 3.4 Ranking shifts under pipeline standardization <a id='3-4'></a>

Below we compare model rankings under **zero-shot** (the standardized pipeline)
vs **CoT** mode to illustrate how pipeline choices shift comparative conclusions.
The rank shift Δ(m) = rank_zs(m) − rank_cot(m) shows which models are most affected.

In [ ]:
bench_scores = df.groupby(['model_label', 'benchmark'])[['zero_shot', 'cot']].mean().reset_index()

benchmarks = sorted(bench_scores['benchmark'].unique())
shift_records = []
for bench in benchmarks:
    b = bench_scores[bench_scores['benchmark'] == bench].set_index('model_label')
    b['rank_zs']  = b['zero_shot'].rank(ascending=False).astype(int)
    b['rank_cot'] = b['cot'].rank(ascending=False).astype(int)
    b['delta']    = b['rank_zs'] - b['rank_cot']
    b['benchmark'] = bench
    shift_records.append(b.reset_index()[['model_label', 'benchmark', 'rank_zs', 'rank_cot', 'delta']])

shifts = pd.concat(shift_records)
pivot_shifts = shifts.pivot_table(index='model_label', columns='benchmark', values='delta')

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    pivot_shifts, cmap='RdBu_r', center=0, vmin=-5, vmax=5,
    annot=True, fmt='.0f', annot_kws={'size': 9},
    linewidths=0.4, ax=ax
)
ax.set_title('Rank shift: zero-shot rank − CoT rank. Positive = model moves up under ZS.')
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout(); plt.show()

large_shifts = (pivot_shifts.abs() >= 2).any(axis=1).sum()
print(f'Models shifting ≥2 ranks on at least one benchmark: {large_shifts}/{len(pivot_shifts)}')

---
## 4. F1(A): Logprob vs Generative Scoring on RedSage-Bench <a id='4-logprob'></a>

RedSage-Bench was evaluated twice for each model: once using **log-probability scoring**
(lighteval's native mode — the model assigns probabilities to A/B/C/D) and once using
**generative scoring** (model generates text, then exact-match or prefix-match extracts
the answer). The two methods measure different things and can disagree substantially.

In [ ]:
SUBSETS = [
    'cybersecurity_knowledge_frameworks',
    'cybersecurity_knowledge_generals',
    'cybersecurity_skills',
    'cybersecurity_tools_cli',
    'cybersecurity_tools_kali',
]

def extract_lighteval_scores(results_dir: Path) -> dict:
    """Pull acc scores from a lighteval results JSON."""
    jsons = list(results_dir.rglob('results_*.json'))
    if not jsons:
        return {}
    d = json.load(open(jsons[0]))
    results = d.get('results', {})
    scores = {}
    for key, vals in results.items():
        m = re.search(r':([^|]+)\|', key)
        if m:
            subset = m.group(1)
            score = vals.get('acc', vals.get('pem', None))
            if score is not None:
                scores[subset] = round(score * 100, 1)
    return scores

lp_rows = []
for d in sorted(OUTPUTS.iterdir()):
    name = d.name
    if name.startswith('redsage_lighteval_') and name.endswith('_full') and 'generative' not in name:
        model_name = name.replace('redsage_lighteval_', '').replace('_full', '')
        lp_scores  = extract_lighteval_scores(d)
        gen_dir    = OUTPUTS / (name.replace('_full', '_generative_full'))
        gen_scores = extract_lighteval_scores(gen_dir) if gen_dir.exists() else {}
        for subset in SUBSETS:
            lp  = lp_scores.get(subset)
            gen = gen_scores.get(subset)
            if lp is not None and gen is not None:
                lp_rows.append({'model': model_name, 'subset': subset, 'logprob': lp, 'generative': gen})

comp = pd.DataFrame(lp_rows)
comp['delta'] = comp['logprob'] - comp['generative']
print(f'Loaded {comp["model"].nunique()} models across {comp["subset"].nunique()} subsets')
comp.head()

In [ ]:
agg = comp.groupby('model')[['logprob', 'generative', 'delta']].mean().round(1)
agg = agg.sort_values('delta', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x = np.arange(len(agg)); w = 0.35
ax.bar(x - w/2, agg['logprob'],    w, label='Log-prob',   color='#4C72B0')
ax.bar(x + w/2, agg['generative'], w, label='Generative', color='#DD8452')
ax.set_xticks(x)
ax.set_xticklabels(agg.index, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Mean accuracy (%) across RedSage subsets')
ax.set_title('Log-prob vs Generative scoring')
ax.legend()

ax = axes[1]
bar_colors = ['#d62728' if v < 0 else '#2ca02c' for v in agg['delta']]
ax.barh(agg.index, agg['delta'], color=bar_colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Log-prob − Generative (pp)')
ax.set_title('Score gap: positive = logprob higher')

plt.tight_layout(); plt.show()
print(agg.to_string())

In [ ]:
pivot_lp  = comp.pivot_table(index='model', columns='subset', values='logprob').round(1)
pivot_gen = comp.pivot_table(index='model', columns='subset', values='generative').round(1)
pivot_d   = comp.pivot_table(index='model', columns='subset', values='delta').round(1)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
kw = dict(annot=True, fmt='.0f', annot_kws={'size': 7}, linewidths=0.3, vmin=0, vmax=100)
sns.heatmap(pivot_lp,  cmap='Blues',   ax=axes[0], **kw); axes[0].set_title('Log-prob accuracy')
sns.heatmap(pivot_gen, cmap='Oranges', ax=axes[1], **kw); axes[1].set_title('Generative accuracy')
kw2 = dict(annot=True, fmt='.0f', annot_kws={'size': 7}, linewidths=0.3,
           cmap='RdBu_r', center=0, vmin=-50, vmax=50)
sns.heatmap(pivot_d, ax=axes[2], **kw2); axes[2].set_title('Delta (logprob − generative)')
for ax in axes:
    ax.set_xlabel(''); ax.set_ylabel('')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=7)
plt.tight_layout(); plt.show()

---
## 5. Failure Mode Examples <a id='5-failures'></a>

The most impactful failure modes are best understood through concrete examples.
Here we pull actual model outputs and scoring decisions from the per-sample detail files.

### 5.1 F2(I): Token-budget filter — SecEval <a id='5-1'></a>

SecEval's official evaluation specifies a **5-token output budget**. This is enough
for a model that immediately outputs `ABC` but insufficient for any preamble.
For one API backend, 5 tokens fell below the server minimum, causing every request
to fail silently. Raising the budget to the backend minimum recovered **81.4%**.

In [ ]:
def load_detail(model_dir: Path, task: str) -> pd.DataFrame:
    path = model_dir / f'{task}_detail.jsonl'
    if not path.exists():
        return pd.DataFrame()
    rows = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    return pd.DataFrame(rows)

eval_base = OUTPUTS / 'eval_results'

seceval_rows = []
for model_dir in sorted(eval_base.iterdir()):
    if not model_dir.is_dir(): continue
    result_path = model_dir / 'seceval_result.json'
    if not result_path.exists(): continue
    r = json.load(open(result_path))
    seceval_rows.append({
        'model': model_dir.name,
        'accuracy': r.get('primary_score', r.get('original_result', {}).get('accuracy_percent')),
        'correct':  r.get('correct',  r.get('original_result', {}).get('correct')),
        'total':    r.get('total',    r.get('original_result', {}).get('total_rows')),
        'invalid':  r.get('invalid',  r.get('original_result', {}).get('invalid')),
    })

se_df = pd.DataFrame(seceval_rows).dropna(subset=['accuracy']).sort_values('accuracy', ascending=False)
print('SecEval scores under original harness:')
print(se_df.to_string(index=False))

### 5.2 F2(E): Denominator inflation — CTI-Bench RCM <a id='5-2'></a>

In CTI-Bench's root-cause mapping task, the **original evaluator excludes invalid
predictions from the denominator** (correct-over-valid scoring). If a model produces
almost no parseable outputs, the few correct ones can yield near-100% accuracy —
while correct-over-total would give near-0%.

We show this by computing both denominators on the same detail file.

In [ ]:
denom_rows = []
for model_dir in sorted(eval_base.iterdir()):
    if not model_dir.is_dir(): continue
    detail = load_detail(model_dir, 'rcm')
    if detail.empty: continue
    total   = len(detail)
    valid   = detail['valid'].sum() if 'valid' in detail.columns else total
    correct = detail['correct'].sum() if 'correct' in detail.columns else 0
    denom_rows.append({
        'model': model_dir.name,
        'total': total,
        'valid': int(valid),
        'correct': int(correct),
        'acc_over_valid': round(correct / valid * 100, 1) if valid > 0 else 0,
        'acc_over_total': round(correct / total * 100, 1),
    })

denom_df = pd.DataFrame(denom_rows).sort_values('acc_over_valid', ascending=False)
denom_df['inflation'] = denom_df['acc_over_valid'] - denom_df['acc_over_total']
print('RCM — valid-only vs total denominator:')
print(denom_df[['model', 'total', 'valid', 'correct',
               'acc_over_valid', 'acc_over_total', 'inflation']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(denom_df)); w = 0.35
ax.bar(x - w/2, denom_df['acc_over_valid'], w, label='Correct / valid (original)',     color='#4C72B0')
ax.bar(x + w/2, denom_df['acc_over_total'], w, label='Correct / total (standardized)', color='#DD8452')
ax.set_xticks(x)
ax.set_xticklabels(denom_df['model'], rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Accuracy (%)')
ax.set_title('F2(E): Denominator policy impact on RCM accuracy')
ax.legend(); plt.tight_layout(); plt.show()

### 5.3 F3(E): Metric-direction mismatch — VSP <a id='5-3'></a>

CTI-Bench and AthenaBench both evaluate CVSS vulnerability scoring, but use
**opposing conventions**: CTI-Bench reports mean absolute deviation (MAD, lower = better),
while AthenaBench normalises the same quantity into a percentage (higher = better).
Aggregating both as "accuracy" produces a semantically inconsistent benchmark score.

In [ ]:
vsp_rows = []
for model_dir in sorted(eval_base.iterdir()):
    if not model_dir.is_dir(): continue
    for task, metric_label, higher_better in [
        ('vsp',        'CTI-Bench VSP (MAD, ↓)',  False),
        ('athena_vsp', 'AthenaBench VSP (%, ↑)',   True),
    ]:
        rp = model_dir / f'{task}_result.json'
        if not rp.exists(): continue
        r = json.load(open(rp))
        score = r.get('primary_score', r.get('original_result', {}).get('accuracy_percent'))
        if score is not None:
            vsp_rows.append({'model': model_dir.name, 'task': metric_label,
                             'score': score, 'higher_better': higher_better})

vsp_df = pd.DataFrame(vsp_rows)
if not vsp_df.empty:
    pivot_vsp = vsp_df.pivot_table(index='model', columns='task', values='score').round(2)
    print('VSP scores by convention:')
    print(pivot_vsp.to_string())

    for col in pivot_vsp.columns:
        higher = 'AthenaBench' in col
        pivot_vsp[col + '_rank'] = pivot_vsp[col].rank(ascending=not higher).astype(int)
    rank_cols  = [c for c in pivot_vsp.columns if 'rank' in c]
    score_cols = [c for c in pivot_vsp.columns if 'rank' not in c]
    if len(rank_cols) == 2:
        pivot_vsp['rank_inversion'] = (pivot_vsp[rank_cols[0]] - pivot_vsp[rank_cols[1]]).abs()
        print('\nRank inversions from metric-direction mismatch:')
        print(
            pivot_vsp[score_cols + rank_cols + ['rank_inversion']]
            .sort_values('rank_inversion', ascending=False)
            .to_string()
        )
else:
    print('VSP result files not found in eval_results — check model dir names.')